# Goal Class Classification

Predict `goal_class` (the goals bucket for a team's **next** match) from the current match's stats.

**Important:** `goal_class` is derived from `next_match_goals`, so both `next_match_goals` and `goals`-of-the-next-match must be kept out of the features to avoid target leakage. We only use the *current* match's stats as predictors.

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)

In [21]:
# Load the prepared data
df = pd.read_csv("cleaneddata.csv", index_col=0)

# Rows where the team had no next match have no target -> drop them
df = df.dropna(subset=["goal_class"]).reset_index(drop=True)

print("Shape:", df.shape)
print(df["goal_class"].value_counts())
df.head()

Shape: (740, 44)
goal_class
0-1    445
2+     295
Name: count, dtype: int64


,table_position,team_name,matchday,total_shots,shots_on_target,goals,possesion,duels_won,opponent_goals,opponent_possession,...,roll3_shot_efficiency,roll5_shot_efficiency,prev_goals,prev_shots_on_target,prev_shot_efficiency,prev_possesion,season_avg_goals,season_avg_efficiency,goals_trend,efficiency_trend
0,2,Arsenal,1,10,4,2,0.43,0.58,0.0,0.57,...,0.500000,0.500000,2.5,5.578947,0.411351,0.585,2.00,0.500000,0.000000,0.000000
1,2,Arsenal,2,15,7,4,0.49,0.52,2.0,0.51,...,0.535714,0.535714,2.0,4.000000,0.500000,0.430,3.00,0.535714,0.000000,0.000000
2,2,Arsenal,3,14,6,3,0.59,0.51,0.0,0.41,...,0.523810,0.523810,4.0,7.000000,0.571429,0.490,3.00,0.523810,0.000000,0.000000
3,2,Arsenal,4,22,8,4,0.72,0.48,1.0,0.28,...,0.523810,0.517857,3.0,6.000000,0.500000,0.590,3.25,0.517857,0.416667,0.005952
4,2,Arsenal,5,22,9,4,0.59,0.53,1.0,0.41,...,0.481481,0.503175,4.0,8.000000,0.500000,0.720,3.40,0.503175,0.266667,-0.021693


In [23]:
# Target
y = df["goal_class"]

# Features: use current-match stats only.
# Drop the target itself and next_match_goals (the column goal_class was derived
# from) to avoid leaking the answer into the model.
leakage_cols = ["goal_class", "next_match_goals"]
X = df.drop(columns=leakage_cols)

# Identify column types for preprocessing.
# After the opponent merge, 'opponent' is also a team-name string and must be
# treated as categorical, otherwise StandardScaler tries to cast it to float.
categorical_cols = [c for c in ["team_name", "opponent"] if c in X.columns]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

print("Features used:", list(X.columns))
print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

Features used: ['table_position', 'team_name', 'matchday', 'total_shots', 'shots_on_target', 'goals', 'possesion', 'duels_won', 'opponent_goals', 'opponent_possession', 'opponent_total_shots', 'opponent_shots_on_target', 'opponent_duels_won', 'is_home', 'shot_efficiency', 'opp_str_avg_goals_scored', 'opp_str_avg_shots_on_target', 'opp_str_avg_possesion', 'opp_str_table_position', 'opp_str_avg_goals_conceded', 'rolling_avg_goals', 'rolling_avg_efficiency', 'roll3_goals', 'roll5_goals', 'roll3_shots_on_target', 'roll5_shots_on_target', 'roll3_total_shots', 'roll5_total_shots', 'roll3_possesion', 'roll5_possesion', 'roll3_duels_won', 'roll5_duels_won', 'roll3_shot_efficiency', 'roll5_shot_efficiency', 'prev_goals', 'prev_shots_on_target', 'prev_shot_efficiency', 'prev_possesion', 'season_avg_goals', 'season_avg_efficiency', 'goals_trend', 'efficiency_trend']
Numeric: ['table_position', 'matchday', 'total_shots', 'shots_on_target', 'goals', 'possesion', 'duels_won', 'opponent_goals', 'oppo

In [24]:
# Train/test split, stratified so both classes keep their proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (592, 42)  Test: (148, 42)


In [25]:
# Preprocessing: scale numeric features, one-hot encode team_name.
# Wrapping this in a pipeline per-model means scaling is fit on the training
# fold only, which keeps the evaluation honest.
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

In [31]:
# A handful of different sklearn classifiers to compare
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Naive Bayes": GaussianNB(),
}

In [35]:
# Train and evaluate each model
results = []
fitted = {}

for name, clf in models.items():
    pipe = Pipeline(steps=[("prep", preprocess), ("model", clf)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe

    preds = pipe.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="weighted")

    # 5-fold cross-validation on the training set for a more stable estimate
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy").mean()

    results.append({"model": name, "test_accuracy": acc, "test_f1": f1, "cv_accuracy": cv})

results_df = pd.DataFrame(results).sort_values("test_f1", ascending=False).reset_index(drop=True)

In [ ]:
# Detailed report + confusion matrix for the best model (by weighted F1)
best_name = results_df.iloc[0]["model"]
best_model = fitted[best_name]
best_preds = best_model.predict(X_test)

print(f"Best model: {best_name}\n")
report = classification_report(y_test, best_preds, output_dict=True)

print("Accuracy: ", accuracy_score(y_test, best_preds))
print("Precision: ", report["weighted avg"]["precision"])
print("Recall: ", report["weighted avg"]["recall"])
print("F1 Score: ", report["weighted avg"]["f1-score"])

Best model: Naive Bayes

Accuracy: 0.8403513513513513
Precision: 0.8313513513513513
Recall: 0.8413513513513513
F1 Score: 0.8394553044517014
